 # 02 — Baseline

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src import config

BOOKS_DIR = config.RAW_DIR / "books"

In [2]:
sample = pd.read_csv(config.INTERIM_DIR / "sample.csv")
sample["text"] = [
    (BOOKS_DIR / f"{book_id}.txt").read_text(encoding="utf-8", errors="replace")
    for book_id in sample["id"]
]
sample["n_words"] = sample["text"].str.split().str.len()
sample[["title", "period", "n_words"]].head()

,title,period,n_words
0,Halleck's New English Literature by Reuben P. ...,pre-1750,167833
1,The Alchemist by Ben Jonson,pre-1750,48264
2,A Short View of the Immorality and Profaneness...,pre-1750,62215
3,Poems of Henry Vaughan Silurist Volume II by H...,pre-1750,71940
4,The Submarine Boys for the Flag by Victor G. D...,pre-1750,46602


Some of these books are labelled incorrectly on the timeframe they were published. Will need to investigate.

In [3]:
import re

def clean_text(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)                    # remove anything in [square brackets]
    text = re.sub(r"FOOTNOTES?:", " ", text, flags=re.IGNORECASE)  # remove "FOOTNOTE:" / "FOOTNOTES:" headings
    return text

sample["text"] = sample["text"].apply(clean_text)
sample["n_words"] = sample["text"].str.split().str.len()

Cleaning the sources of footnotes.

In [4]:
print(sample["text"].iloc[1][:1500])





Produced by Amy E Zelmer, Robert Prince, Sue Asscher





THE ALCHEMIST


By Ben Jonson




INTRODUCTION

The greatest of English dramatists except Shakespeare, the first
literary dictator and poet-laureate, a writer of verse, prose,
satire, and criticism who most potently of all the men of his time
affected the subsequent course of English letters: such was Ben
Jonson, and as such his strong personality assumes an interest to
us almost unparalleled, at least in his age.

Ben Jonson came of the stock that was centuries after to give to
the world Thomas Carlyle; for Jonson's grandfather was of
Annandale, over the Solway, whence he migrated to England.
Jonson's father lost his estate under Queen Mary, "having been cast
into prison and forfeited."  He entered the church, but died a
month before his illustrious son was born, leaving his widow and
child in poverty.  Jonson's birthplace was Westminster, and the
time of his birth early in 1573.  He was thus nearly ten years
Shakespeare's 

In [5]:
print(sample["n_words"].describe())
sample.groupby("period")["n_words"].median()

count       500.000000
mean      70706.678000
std       77394.954804
min         442.000000
25%       20318.000000
50%       50598.500000
75%       94135.500000
max      818001.000000
Name: n_words, dtype: float64


period
1750-1799    39565.5
1800-1849    56352.5
1850-1899    65337.0
1900-1918    55329.5
pre-1750     29846.5
Name: n_words, dtype: float64

No Gutenberg licence. Introductions written after the original publication.

Book lengths vary a lot, and by period, model could learn "short book = old" instead of learning the language. That's the shortcut we want to avoid, since it does not answer our research question. Possible fix is taking a chucnk out of the middle of every book. Would have to to take into account possible instances of notes added after publication. Also solves the issues of introductions/forewords written after publication.

In [6]:
(sample["n_words"] < 5000).sum()

np.int64(23)

In [7]:
sample.loc[sample["n_words"] < 5000, "period"].value_counts()

period
1800-1849    9
1750-1799    8
1850-1899    3
pre-1750     2
1900-1918    1
Name: count, dtype: int64

Checking which books have less than 5000 words, which is the amount we want to sample 

In [8]:
sample = sample[sample["n_words"] >= 5000].reset_index(drop=True)
sample["period"].value_counts().sort_index()

period
1750-1799    92
1800-1849    91
1850-1899    97
1900-1918    99
pre-1750     98
Name: count, dtype: int64

Removing works in which are shorter than 5000 words

In [9]:
sample["period"].value_counts().sort_index()

period
1750-1799    92
1800-1849    91
1850-1899    97
1900-1918    99
pre-1750     98
Name: count, dtype: int64

In [10]:
def middle_chunk(text, n_words=5000):
    words = text.split()
    if len(words) <= n_words:
        return " ".join(words)
    start = (len(words) - n_words) // 2
    return " ".join(words[start:start + n_words])

sample["excerpt"] = sample["text"].apply(middle_chunk)
sample["excerpt"].str.split().str.len().describe()

count     477.0
mean     5000.0
std         0.0
min      5000.0
25%      5000.0
50%      5000.0
75%      5000.0
max      5000.0
Name: excerpt, dtype: float64

creates a small reusable tool. It splits a book into words, finds the middle, and keeps 5,000 words from there.

In [11]:
print(sample["excerpt"].iloc[1][:1000])

Not always necessary: The children of perdition are oft-times Made instruments even of the greatest works: Beside, we should give somewhat to man's nature, The place he lives in, still about the fire, And fume of metals, that intoxicate The brain of man, and make him prone to passion. Where have you greater atheists than your cooks? Or more profane, or choleric, than your glass-men? More antichristian than your bell-founders? What makes the devil so devilish, I would ask you, Sathan, our common enemy, but his being Perpetually about the fire, and boiling Brimstone and arsenic? We must give, I say, Unto the motives, and the stirrers up Of humours in the blood. It may be so, When as the work is done, the stone is made, This heat of his may turn into a zeal, And stand up for the beauteous discipline, Against the menstruous cloth and rag of Rome. We must await his calling, and the coming Of the good spirit. You did fault, t' upbraid him With the brethren's blessing of Heidelberg, weighing 

Checked an excerpt by eye: The Alchemist excerpt is original 1610 play text, not the modern introduction

In [12]:
markers = ["[Footnote", "Footnote", "Transcriber", "Editor's note", "[Note"]
for m in markers:
    n = sample["excerpt"].str.contains(m, case=False, regex=False).sum()
    print(f"{m!r}: {n} excerpts")

'[Footnote': 0 excerpts
'Footnote': 7 excerpts
'Transcriber': 1 excerpts
"Editor's note": 0 excerpts
'[Note': 0 excerpts


Notes within the text eg. footnotes, editors note. Cleaned up at start.

In [13]:
ex = sample[sample["excerpt"].str.contains("Footnote", case=False)].iloc[0]
i = ex["excerpt"].lower().find("footnote")
print(ex["title"], "|", ex["period"])
print(ex["excerpt"][i - 200 : i + 400])

A Vindication of Rights of Woman by Mary Wollstonecraft | 1750-1799
seriously on this curious passage, and I could produce many similar ones; and some, so very sentimental, that I have heard rational men use the word indecent, when they mentioned them with disgust. (*Footnote. Can you?--Can you? would be the most emphatical comment, were it drawled out in a whining voice.) Throughout there is a display of cold, artificial feelings, and that parade of sensibility which boys and girls should be taught to despise as the sure mark of a little vain mind. Florid appeals are made to heaven, and to the BEAUTEOUS INNOCENTS, the fairest images of heaven here below, whil


Older books may have Latin excerpts. 

In [14]:
author = sample["title"].str.extract(r".*\sby\s(.+)$", flags=2)[0].str.strip()

unknown = author.isna() | author.str.lower().isin(["anonymous", "various", "unknown"])
sample["group"] = author.where(~unknown, sample["title"])

sample["group"].value_counts().head(10)

group
Ben Jonson             5
Honore de Balzac       4
Molire                 3
William Shakespeare    3
Anthony Trollope       3
David Hume             2
Joseph Trapp           2
Samuel Johnson         2
Daniel Defoe           2
Henry Fielding         2
Name: count, dtype: int64

In [15]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)
train_idx, test_idx = next(splitter.split(sample, sample["period"], groups=sample["group"]))

train = sample.iloc[train_idx]
test = sample.iloc[test_idx]
print("Train:", len(train), " Test:", len(test))

Train: 382  Test: 95


In [16]:
print("Authors in both train and test:", len(set(train["group"]) & set(test["group"])))
pd.DataFrame({
    "train": train["period"].value_counts(normalize=True),
    "test": test["period"].value_counts(normalize=True),
}).sort_index().round(2)

Authors in both train and test: 0


,train,test
period,,
1750-1799,0.19,0.19
1800-1849,0.19,0.19
1850-1899,0.20,0.20
1900-1918,0.21,0.21
pre-1750,0.20,0.21


In [17]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate

baseline = make_pipeline(
    TfidfVectorizer(),
    DummyClassifier(strategy="most_frequent"),
)

cv = StratifiedGroupKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
scores = cross_validate(
    baseline, train["excerpt"], train["period"],
    groups=train["group"], cv=cv, scoring=["accuracy", "f1_macro"],
)

print(f"Accuracy: {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
print(f"Macro-F1: {scores['test_f1_macro'].mean():.3f} ± {scores['test_f1_macro'].std():.3f}")

Accuracy: 0.202 ± 0.005
Macro-F1: 0.067 ± 0.001


Scoring the baseline

The accuracy is 0.202 as one one period out of 5 is guessed correctly across the 5 folds.
Macro-F1 is very low as 4 out of the 5 are incorrect.
The score barely changes across the 5 folds so the result is stable and trustworthy.
The bar for Milestone 3 is to beat the 0.2 accuracy and the 0.067 macro-F1 to show it has learned from the language.